#### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score, mean_absolute_error
from scipy.stats import wilcoxon

import warnings; warnings.filterwarnings('ignore')

df_model = pd.read_csv('data/df_model_with_ids.csv')

In [ ]:
df_model.head()

In [ ]:
df_model.columns

In [ ]:
df_model = pd.get_dummies(df_model, columns=['gender', 'main_topic'], drop_first=True)

In [ ]:
print(df_model.shape)
print(df_model['open'].isna().sum())
print(df_model['click'].isna().sum())

In [ ]:
df_model = df_model.dropna(subset=['open', 'click'])
print(df_model.shape)

### Temporal instability analysis

In [ ]:
def plot_engagement_over_time(df, mailing_id_col='mailing_id', n_bins=20):
    sorted_ids = np.sort(df[mailing_id_col].unique())
    id_bins = np.array_split(sorted_ids, n_bins)
    
    rows = []
    for idx, bin_ids in enumerate(id_bins):
        subset = df[df[mailing_id_col].isin(bin_ids)]
        rows.append({
            'batch': idx + 1,
            'open_rate': subset['open'].mean(),
            'click_rate': subset['click'].mean(),
            'n_mailings': len(bin_ids)
        })
    
    stability_df = pd.DataFrame(rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(stability_df['batch'], stability_df['open_rate'] * 100,
                 marker='o', color='steelblue')
    axes[0].set_title('Open Rate across Campaign Batches\n(ordered by mailing ID)')
    axes[0].set_xlabel('Batch (chronological)')
    axes[0].set_ylabel('Open Rate (%)')
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)
    
    axes[1].plot(stability_df['batch'], stability_df['click_rate'] * 100,
                 marker='o', color='coral')
    axes[1].set_title('Click Rate across Campaign Batches\n(ordered by mailing ID)')
    axes[1].set_xlabel('Batch (chronological)')
    axes[1].set_ylabel('Click Rate (%)')
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)
    
    plt.suptitle('Temporal Instability of Engagement Rates', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\nEngagement stability statistics:")
    print(f"  Open rate std across batches:  {stability_df['open_rate'].std():.4f}")
    print(f"  Click rate std across batches: {stability_df['click_rate'].std():.4f}")
    print(f"  Open rate range: {stability_df['open_rate'].min():.3f} – {stability_df['open_rate'].max():.3f}")
    
    return stability_df

In [ ]:
stability_df = plot_engagement_over_time(df_model)

# Open model: split on Mailing IDs

#### Batch Split

In [ ]:
def create_batches(df, mailing_id_col='mailing_id', n_batches=10):
    sorted_mailings = np.sort(df[mailing_id_col].unique())
    return np.array_split(sorted_mailings, n_batches)

def get_train_test_for_batch(df, batches, batch_idx, window='expanding', window_size=3):
    test_ids = batches[batch_idx]
    if window == 'expanding':
        train_ids = np.concatenate(batches[:batch_idx])
    else:
        start = max(0, batch_idx - window_size)
        train_ids = np.concatenate(batches[start:batch_idx])
    train = df[df['mailing_id'].isin(train_ids)].copy()
    test = df[df['mailing_id'].isin(test_ids)].copy()
    return train, test

In [ ]:
#Naive baseline prediction function
def naive_baseline_predict(train, test, target='open', user_col='user_id'):
    user_open_rate = train.groupby(user_col)[target].mean()
    global_mean = train[target].mean()
    return test[user_col].map(user_open_rate).fillna(global_mean).values

def get_feature_cols(df):
    exclude = {'user_id', 'mailing_id', 'open', 'click'}
    return [c for c in df.columns if c not in exclude]

In [ ]:
#Static logistic regression prediction function
def train_lr(train, feature_cols, target='open'):
    X_train = train[feature_cols].fillna(0)
    y_train = train[target]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    cw = 'balanced' if target == 'click' else None
    model = LogisticRegression(penalty='l2', C=1.0, class_weight=cw, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model, scaler

def predict_lr(model, scaler, test, feature_cols):
    X_test = test[feature_cols].fillna(0)
    return model.predict_proba(scaler.transform(X_test))[:, 1]


#### Main Loop 

For each batch step it:

Trains the naive baseline on all prior data
Uses the static LR frozen from batch 0 (never updated)
Retrains the adaptive LR on all data seen so far
Records AUC and MAE for all three on the current test batch

In [ ]:
def run_adaptive_evaluation(df, n_batches=10, window='expanding', window_size=3):
    feature_cols = get_feature_cols(df)
    batches = create_batches(df, n_batches=n_batches)

    static_train = df[df['mailing_id'].isin(batches[0])].copy()
    static_fitted = static_train['open'].nunique() == 2
    if static_fitted:
        static_model, static_scaler = train_lr(static_train, feature_cols, 'open')

    results = []
    for i in range(1, len(batches)):
        train, test = get_train_test_for_batch(df, batches, i, window=window, window_size=window_size)
        if len(train) == 0 or len(test) == 0 or test['open'].nunique() < 2:
            continue
        y_test = test['open'].values
        row = {'batch': i, 'n_train_rows': len(train), 'n_test_rows': len(test)}

        naive_preds = naive_baseline_predict(train, test)
        row['baseline_auc'] = roc_auc_score(y_test, naive_preds)
        row['baseline_mae'] = mean_absolute_error(y_test, naive_preds)

        if static_fitted:
            static_preds = predict_lr(static_model, static_scaler, test, feature_cols)
            row['static_lr_auc'] = roc_auc_score(y_test, static_preds)
            row['static_lr_mae'] = mean_absolute_error(y_test, static_preds)

        if train['open'].nunique() == 2:
            adaptive_model, adaptive_scaler = train_lr(train, feature_cols, 'open')
            adaptive_preds = predict_lr(adaptive_model, adaptive_scaler, test, feature_cols)
            row['adaptive_lr_auc'] = roc_auc_score(y_test, adaptive_preds)
            row['adaptive_lr_mae'] = mean_absolute_error(y_test, adaptive_preds)

        results.append(row)
        print(f"Batch {i:2d} | Baseline AUC={row['baseline_auc']:.4f} | Static AUC={row.get('static_lr_auc', float('nan')):.4f} | Adaptive AUC={row.get('adaptive_lr_auc', float('nan')):.4f}")

    return pd.DataFrame(results)

## Evaluation

### Results with expanding window

In [ ]:
results_expanding = run_adaptive_evaluation(df_model, n_batches=10, window='expanding')
results_expanding

### Results with sliding window

This retrains on only the last 3 batches instead of all history 

In [ ]:
results_sliding = run_adaptive_evaluation(df_model, n_batches=10, window='sliding', window_size=3)
results_sliding

### Batch size sensitivity

In [ ]:
all_results = {}
for n in [5, 10, 15, 20]:
    print(f"\n=== n_batches = {n} ===")
    all_results[n] = run_adaptive_evaluation(df_model, n_batches=n, window='expanding')

In [ ]:
for n, res in all_results.items():
    print(f"\nn_batches={n}")
    print(f"  Mean adaptive AUC: {res['adaptive_lr_auc'].mean():.4f}")
    print(f"  Mean static AUC:   {res['static_lr_auc'].mean():.4f}")
    print(f"  Mean baseline AUC: {res['baseline_auc'].mean():.4f}")
    wins = (res['adaptive_lr_auc'] > res['static_lr_auc']).sum()
    print(f"  Adaptive beats static in {wins}/{len(res)} batches")

# Open model: Time split

In [ ]:
mailing_dates = pd.read_csv('data/mailing_dates.csv')

df_model_dated = df_model.merge(mailing_dates, on='mailing_id', how='left')
df_model_dated = df_model_dated.dropna(subset=['send_date'])

print(df_model_dated.shape)
print(df_model_dated['mailing_id'].nunique(), "unique mailings")

#### Batch split 

In [ ]:
def create_batches_by_date(df, date_col='send_date', n_batches=10):
    df_sorted = df[[date_col, 'mailing_id']].drop_duplicates().sort_values(date_col)
    sorted_mailings = df_sorted['mailing_id'].values
    return np.array_split(sorted_mailings, n_batches)

In [ ]:
#Main loop
def run_adaptive_evaluation_dated(df, n_batches=10, window='expanding', window_size=3):
    batches = create_batches_by_date(df, n_batches=n_batches)
    feature_cols = get_feature_cols(df.drop(columns=['send_date']))
    
    static_train = df[df['mailing_id'].isin(batches[0])].copy()
    static_fitted = static_train['open'].nunique() == 2
    if static_fitted:
        static_model, static_scaler = train_lr(static_train, feature_cols, 'open')

    results = []
    for i in range(1, len(batches)):
        train, test = get_train_test_for_batch(df, batches, i, window=window, window_size=window_size)
        if len(train) == 0 or len(test) == 0 or test['open'].nunique() < 2:
            continue
        y_test = test['open'].values
        row = {'batch': i, 'n_train_rows': len(train), 'n_test_rows': len(test)}

        naive_preds = naive_baseline_predict(train, test)
        row['baseline_auc'] = roc_auc_score(y_test, naive_preds)
        row['baseline_mae'] = mean_absolute_error(y_test, naive_preds)

        if static_fitted:
            static_preds = predict_lr(static_model, static_scaler, test, feature_cols)
            row['static_lr_auc'] = roc_auc_score(y_test, static_preds)
            row['static_lr_mae'] = mean_absolute_error(y_test, static_preds)

        if train['open'].nunique() == 2:
            adaptive_model, adaptive_scaler = train_lr(train, feature_cols, 'open')
            adaptive_preds = predict_lr(adaptive_model, adaptive_scaler, test, feature_cols)
            row['adaptive_lr_auc'] = roc_auc_score(y_test, adaptive_preds)
            row['adaptive_lr_mae'] = mean_absolute_error(y_test, adaptive_preds)

        results.append(row)
        print(f"Batch {i:2d} | Baseline AUC={row['baseline_auc']:.4f} | Static AUC={row.get('static_lr_auc', float('nan')):.4f} | Adaptive AUC={row.get('adaptive_lr_auc', float('nan')):.4f}")

    return pd.DataFrame(results)

### Results with expanding window

In [ ]:
results_dated = run_adaptive_evaluation_dated(df_model_dated, n_batches=10, window='expanding')
results_dated

In [ ]:
print(results_dated[['baseline_auc','static_lr_auc','adaptive_lr_auc']].mean())

# Click model

In [ ]:
#Main loop
def run_adaptive_evaluation_click(df, n_batches=10, window='expanding', window_size=3):
    feature_cols = get_feature_cols(df)
    batches = create_batches(df, n_batches=n_batches)
 
    static_train = df[df['mailing_id'].isin(batches[0])].copy()
    static_fitted = static_train['click'].nunique() == 2
    if static_fitted:
        static_model, static_scaler = train_lr(static_train, feature_cols, 'click')
 
    results = []
    for i in range(1, len(batches)):
        train, test = get_train_test_for_batch(df, batches, i, window=window, window_size=window_size)
        if len(train) == 0 or len(test) == 0 or test['click'].nunique() < 2:
            print(f"Batch {i}: test set has only one class for click, skipping.")
            continue
 
        y_test = test['click'].values
        row = {'batch': i, 'n_train_rows': len(train), 'n_test_rows': len(test)}
 
       
        naive_preds = naive_baseline_predict(train, test, target='click')
        row['baseline_auc'] = roc_auc_score(y_test, naive_preds)
        row['baseline_prauc'] = average_precision_score(y_test, naive_preds)
 
        if static_fitted:
            static_preds = predict_lr(static_model, static_scaler, test, feature_cols)
            row['static_lr_auc'] = roc_auc_score(y_test, static_preds)
            row['static_lr_prauc'] = average_precision_score(y_test, static_preds)
 
        
        if train['click'].nunique() == 2:
            adaptive_model, adaptive_scaler = train_lr(train, feature_cols, 'click')
            adaptive_preds = predict_lr(adaptive_model, adaptive_scaler, test, feature_cols)
            row['adaptive_lr_auc'] = roc_auc_score(y_test, adaptive_preds)
            row['adaptive_lr_prauc'] = average_precision_score(y_test, adaptive_preds)
        else:
            row['adaptive_lr_auc'] = float('nan')
            row['adaptive_lr_prauc'] = float('nan')
 
        results.append(row)
        print(f"Batch {i:2d} | Baseline AUC={row['baseline_auc']:.4f} PR-AUC={row['baseline_prauc']:.4f} "
              f"| Static AUC={row.get('static_lr_auc', float('nan')):.4f} PR-AUC={row.get('static_lr_prauc', float('nan')):.4f} "
              f"| Adaptive AUC={row.get('adaptive_lr_auc', float('nan')):.4f} PR-AUC={row.get('adaptive_lr_prauc', float('nan')):.4f}")
 
    return pd.DataFrame(results)

### Results with expanding window

In [ ]:
results_click = run_adaptive_evaluation_click(df_model, n_batches=10, window='expanding')
results_click

In [ ]:
print(results_click[['baseline_auc','static_lr_auc','adaptive_lr_auc',
                      'baseline_prauc','static_lr_prauc','adaptive_lr_prauc']].mean())

### Results with sliding window

In [ ]:
results_click_sliding = run_adaptive_evaluation_click(df_model, n_batches=10, window='sliding', window_size=3)
results_click_sliding

In [ ]:
print(results_click_sliding[['baseline_auc','static_lr_auc','adaptive_lr_auc',
                              'baseline_prauc','static_lr_prauc','adaptive_lr_prauc']].mean())

In [ ]:
n_beats_auc = (results_click_sliding['adaptive_lr_auc'] > results_click_sliding['static_lr_auc']).sum()
n_beats_prauc = (results_click_sliding['adaptive_lr_prauc'] > results_click_sliding['static_lr_prauc']).sum()
print(f"Adaptive beats static AUC: {n_beats_auc}/9, PR-AUC: {n_beats_prauc}/9")

### Batch size sensitivity

In [ ]:
all_results_click = {}
for n in [5, 10, 15, 20]:
    print(f"\n=== n_batches = {n} ===")
    all_results_click[n] = run_adaptive_evaluation_click(df_model, n_batches=n, window='expanding')

for n, res in all_results_click.items():
    print(f"\nn_batches={n}")
    print(f"  Mean AUC   - baseline: {res['baseline_auc'].mean():.4f} | static: {res['static_lr_auc'].mean():.4f} | adaptive: {res['adaptive_lr_auc'].mean():.4f}")
    print(f"  Mean PRAUC - baseline: {res['baseline_prauc'].mean():.4f} | static: {res['static_lr_prauc'].mean():.4f} | adaptive: {res['adaptive_lr_prauc'].mean():.4f}")

In [ ]:
for n, res in all_results_click.items():
    print(f"n_batches={n}")
    print(f"  Mean AUC   - baseline: {res['baseline_auc'].mean():.4f} | static: {res['static_lr_auc'].mean():.4f} | adaptive: {res['adaptive_lr_auc'].mean():.4f}")
    print(f"  Mean PRAUC - baseline: {res['baseline_prauc'].mean():.4f} | static: {res['static_lr_prauc'].mean():.4f} | adaptive: {res['adaptive_lr_prauc'].mean():.4f}")
    wins = (res['adaptive_lr_auc'] > res['static_lr_auc']).sum()
    print(f"  Adaptive beats static in {wins}/{len(res)} batches")
    print()

# Summary

#### Functions

In [ ]:
def plot_auc_over_batches(results_df, title_suffix='', window_label='Expanding Window', metric='auc'):
    fig, ax = plt.subplots(figsize=(8, 5))
    steps = results_df['batch']

    baseline_col = f'baseline_{metric}'
    static_col = f'static_lr_{metric}'
    adaptive_col = f'adaptive_lr_{metric}'

    ax.plot(steps, results_df[baseline_col], label='Naive Baseline',
            marker='o', linestyle='--', color='grey')
    if static_col in results_df.columns:
        ax.plot(steps, results_df[static_col], label='Static LR (trained once)',
                marker='s', linestyle=':', color='steelblue')
    ax.plot(steps, results_df[adaptive_col], label='Adaptive LR (retrained)',
            marker='^', linestyle='-', color='coral')

    vals = results_df[adaptive_col].values
    for j in range(1, len(vals)):
        if not np.isnan(vals[j]) and not np.isnan(vals[j-1]):
            if metric in ('auc', 'prauc') and vals[j] < vals[j-1] - 0.01:
                ax.annotate('↓', xy=(steps.iloc[j], vals[j]), fontsize=14, color='red', ha='center', va='top')
            elif metric == 'mae' and vals[j] > vals[j-1] + 0.01:
                ax.annotate('↓', xy=(steps.iloc[j], vals[j]), fontsize=14, color='red', ha='center', va='bottom')

    metric_label = {'auc': 'AUC-ROC', 'mae': 'MAE', 'prauc': 'PR-AUC'}[metric]
    ax.set_title(f'{metric_label} over Retraining Steps ({window_label}){title_suffix}', fontsize=12)
    ax.set_xlabel('Batch Step')
    ax.set_ylabel(metric_label)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


def plot_batch_size_comparison(all_results, metric='auc', target='open'):
    fig, ax = plt.subplots(figsize=(8, 5))

    batch_sizes = sorted(all_results.keys())
    baseline_col = f'baseline_{metric}'
    static_col = f'static_lr_{metric}'
    adaptive_col = f'adaptive_lr_{metric}'

    baseline_means = [all_results[n][baseline_col].mean() for n in batch_sizes]
    static_means = [all_results[n][static_col].mean() for n in batch_sizes]
    adaptive_means = [all_results[n][adaptive_col].mean() for n in batch_sizes]

    ax.plot(batch_sizes, baseline_means, marker='o', linestyle='--', label='Naive Baseline', color='grey')
    ax.plot(batch_sizes, static_means, marker='s', linestyle=':', label='Static LR', color='steelblue')
    ax.plot(batch_sizes, adaptive_means, marker='^', linestyle='-', label='Adaptive LR', color='coral')

    metric_label = {'auc': 'AUC-ROC', 'prauc': 'PR-AUC'}[metric]
    ax.set_xlabel('Number of Batches')
    ax.set_ylabel(f'Mean {metric_label}')
    ax.set_title(f'Effect of Batch Size on Mean {metric_label} ({target})')
    ax.set_xticks(batch_sizes)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


def print_summary_table(results_df, target='open', metrics=('auc', 'mae')):
    cols = ['batch', 'n_train_rows', 'n_test_rows']
    for m in metrics:
        for model in ['baseline', 'static_lr', 'adaptive_lr']:
            col = f'{model}_{m}'
            if col in results_df.columns:
                cols.append(col)

    summary = results_df[cols].round(4)

    print("=" * 90)
    print(f"ADAPTIVE RETRAINING RESULTS SUMMARY — target = {target}")
    print("=" * 90)
    print(summary.to_string(index=False))
    print("=" * 90)

    print("\nMEAN PERFORMANCE ACROSS ALL BATCHES:")
    for model in ['baseline', 'static_lr', 'adaptive_lr']:
        line = f"  {model:12s} | "
        for m in metrics:
            col = f'{model}_{m}'
            if col in results_df.columns:
                line += f"{m.upper()}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}  "
        print(line)

    for m in metrics:
        adaptive_col = f'adaptive_lr_{m}'
        static_col = f'static_lr_{m}'
        baseline_col = f'baseline_{m}'
        if adaptive_col in results_df.columns and static_col in results_df.columns:
            valid = results_df[[adaptive_col, static_col]].dropna()
            if m == 'mae':
                n_beats = (valid[adaptive_col] < valid[static_col]).sum()
            else:
                n_beats = (valid[adaptive_col] > valid[static_col]).sum()
            print(f"\nAdaptive LR beats Static LR on {m.upper()} in {n_beats}/{len(valid)} batches")
        if adaptive_col in results_df.columns and baseline_col in results_df.columns:
            valid = results_df[[adaptive_col, baseline_col]].dropna()
            if m == 'mae':
                n_beats = (valid[adaptive_col] < valid[baseline_col]).sum()
            else:
                n_beats = (valid[adaptive_col] > valid[baseline_col]).sum()
            print(f"Adaptive LR beats Naive Baseline on {m.upper()} in {n_beats}/{len(valid)} batches")
    print()

## Plots

### Open

In [ ]:
# Open: expanding window AUC and MAE 
plot_auc_over_batches(results_expanding, window_label='Expanding Window', metric='auc')
plot_auc_over_batches(results_expanding, window_label='Expanding Window', metric='mae')

# Open: sliding window AUC and MAE
plot_auc_over_batches(results_sliding, window_label='Sliding Window', metric='auc')
plot_auc_over_batches(results_sliding, window_label='Sliding Window', metric='mae')

In [ ]:
# Open: batch size comparison (AUC)
plot_batch_size_comparison(all_results, metric='auc', target='open')

### Click

In [ ]:
# Click: expanding window AUC and PR-AUC
plot_auc_over_batches(results_click, window_label='Expanding Window', metric='auc')
plot_auc_over_batches(results_click, window_label='Expanding Window', metric='prauc')

# Click: sliding window AUC and PR-AUC
plot_auc_over_batches(results_click_sliding, window_label='Sliding Window', metric='auc')
plot_auc_over_batches(results_click_sliding, window_label='Sliding Window', metric='prauc')

In [ ]:
# Click: batch size comparison (AUC and PR-AUC)
plot_batch_size_comparison(all_results_click, metric='auc', target='click')
plot_batch_size_comparison(all_results_click, metric='prauc', target='click')

## Tables

In [ ]:
print_summary_table(results_expanding, target='open', metrics=('auc', 'mae'))

In [ ]:
print_summary_table(results_click, target='click', metrics=('auc', 'prauc'))

## Test

In [ ]:
# Open
diff_open = (results_expanding['adaptive_lr_auc'] - results_expanding['static_lr_auc']).dropna()
stat, p = wilcoxon(diff_open)
print(f"Open AUC — Wilcoxon p-value: {p:.4f}")

# Click
diff_click = (results_click['adaptive_lr_auc'] - results_click['static_lr_auc']).dropna()
stat, p = wilcoxon(diff_click)
print(f"Click AUC — Wilcoxon p-value: {p:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_static_auc_degradation(results_df):
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(results_df['batch'], results_df['static_lr_auc'], 
            marker='s', linestyle='-', color='steelblue', label='Static LR (trained once)')
    
    # Add a horizontal reference line at the first batch AUC
    first_auc = results_df['static_lr_auc'].iloc[0]
    ax.axhline(y=first_auc, linestyle='--', color='grey', alpha=0.6, label=f'Initial AUC (batch 1): {first_auc:.3f}')
    
    # Annotate min point
    min_idx = results_df['static_lr_auc'].idxmin()
    min_batch = results_df.loc[min_idx, 'batch']
    min_auc = results_df.loc[min_idx, 'static_lr_auc']
    ax.annotate(f'Min: {min_auc:.3f}', 
                xy=(min_batch, min_auc), 
                xytext=(min_batch + 0.3, min_auc + 0.01),
                fontsize=9, color='red')
    
    ax.set_title('Static LR AUC Degradation over Time\n(model trained once on earliest batch)', fontsize=12)
    ax.set_xlabel('Batch (chronological)')
    ax.set_ylabel('AUC-ROC')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    print("\nStatic LR AUC statistics:")
    print(f"  Initial AUC (batch 1): {results_df['static_lr_auc'].iloc[0]:.4f}")
    print(f"  Final AUC (last batch): {results_df['static_lr_auc'].iloc[-1]:.4f}")
    print(f"  Min AUC: {results_df['static_lr_auc'].min():.4f}")
    print(f"  Max AUC: {results_df['static_lr_auc'].max():.4f}")
    print(f"  Std: {results_df['static_lr_auc'].std():.4f}")

plot_static_auc_degradation(results_expanding)

# Best models after batch sensitivity (n=20)

## Open

In [ ]:
results_expanding_n20 = run_adaptive_evaluation(df_model, n_batches=20, window='expanding')

plot_auc_over_batches(results_expanding_n20, 
                      window_label='Expanding Window', 
                      metric='auc',
                      title_suffix=' (n_batches=20)')

## Click

In [ ]:
results_click_n20 = run_adaptive_evaluation_click(df_model, n_batches=20, window='expanding')

plot_auc_over_batches(results_click_n20,
                      window_label='Expanding Window',
                      metric='auc',
                      title_suffix=' (n_batches=20)')

plot_auc_over_batches(results_click_n20,
                      window_label='Expanding Window',
                      metric='prauc',
                      title_suffix=' (n_batches=20)')

## Summary

In [ ]:
print_summary_table(results_expanding_n20, target='open', metrics=('auc', 'mae'))

In [ ]:
print_summary_table(results_click_n20, target='click', metrics=('auc', 'prauc'))

In [ ]:
# Wilcoxon tests at n=20 
from scipy.stats import wilcoxon

diff_open_n20 = (results_expanding_n20['adaptive_lr_auc'] - results_expanding_n20['static_lr_auc']).dropna()
stat, p = wilcoxon(diff_open_n20)
print(f"Open AUC Wilcoxon (n=20): stat={stat:.4f}, p={p:.4f}")

diff_click_n20 = (results_click_n20['adaptive_lr_auc'] - results_click_n20['static_lr_auc']).dropna()
stat, p = wilcoxon(diff_click_n20)
print(f"Click AUC Wilcoxon (n=20): stat={stat:.4f}, p={p:.4f}")